# 16 — Climate vs. Sheltered/Unsheltered Homeless Rate (County/CoC)

**Hypothesis:** Colder climates push a larger share of homeless people into shelters without
necessarily reducing the total homeless rate. Warmer cities have high unsheltered rates
because sleeping outside is survivable year-round.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_county_data.csv')
df = df.dropna(subset=['homeless_rate_per_10k', 'unsheltered_pct', 'jan_temp_f'])
print(f'Loaded {len(df)} CoCs with climate data')

Loaded 101 CoCs with climate data


In [2]:
slope, intercept, r, p, se = stats.linregress(df['jan_temp_f'], df['unsheltered_pct'])
r2 = r ** 2
print(f'Jan temp vs unsheltered %: r={r:.3f}, R²={r2:.3f}, p={p:.4f}')

x_range = [df['jan_temp_f'].min(), df['jan_temp_f'].max()]
y_range = [slope * x + intercept for x in x_range]

fig = px.scatter(
    df, x='jan_temp_f', y='unsheltered_pct',
    size='total_homeless',
    color='homeless_rate_per_10k', color_continuous_scale='Reds',
    hover_name='coc_name',
    hover_data={'state_postal': True, 'jan_temp_f': ':.0f', 'unsheltered_pct': ':.1f', 'total_homeless': ':,', 'homeless_rate_per_10k': ':.1f'},
    title=f'January Temperature vs. Unsheltered % of Homeless (CoC Level)<br><sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f} — warmer climates → higher unsheltered share</sup>',
    labels={'jan_temp_f': 'Avg January Temperature (°F)', 'unsheltered_pct': '% Homeless Who Are Unsheltered'},
    size_max=60,
)
fig.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression', line=dict(color='black', dash='dash')))
fig.show()

Jan temp vs unsheltered %: r=0.703, R²=0.494, p=0.0000


In [3]:
slope2, intercept2, r2v, p2, se2 = stats.linregress(df['jan_temp_f'], df['homeless_rate_per_10k'])
r2_2 = r2v ** 2
print(f'Jan temp vs total homeless rate: r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f}')

x_range2 = [df['jan_temp_f'].min(), df['jan_temp_f'].max()]
y_range2 = [slope2 * x + intercept2 for x in x_range2]

fig2 = px.scatter(
    df, x='jan_temp_f', y='homeless_rate_per_10k',
    size='total_homeless',
    color='unsheltered_pct', color_continuous_scale='OrRd',
    hover_name='coc_name',
    title=f'January Temperature vs. Total Homeless Rate (CoC Level)<br><sup>r={r2v:.3f}, R²={r2_2:.3f}, p={p2:.4f}</sup>',
    labels={'jan_temp_f': 'Avg January Temperature (°F)', 'homeless_rate_per_10k': 'Homeless Rate per 10k'},
    size_max=60,
)
fig2.add_trace(go.Scatter(x=x_range2, y=y_range2, mode='lines', name='Regression', line=dict(color='black', dash='dash')))
fig2.show()

Jan temp vs total homeless rate: r=0.181, R²=0.033, p=0.0700


## Climate Hypothesis Testing

**Key findings:**
- **Unsheltered % increases with temperature**: Warmer climates allow year-round outdoor living
- **Total rate weakly associated with temperature**: Suggests climate influences shelter-seeking behavior, not homelessness rate itself
- **Policy implication**: Cold-weather CoCs have lower unsheltered % (forced sheltering); warm-weather CoCs need expanded shelter capacity to reduce visible homelessness
- **Size bubble**: Larger CoCs appear across temperature ranges, indicating homelessness is a structural problem independent of climate